# Residual-PII Leak Test — Azure `o4-mini` judge

Reproducible **Step 2 (the LLM judge)** of the residual-PII leak test
(`evaluation/leak_tests/README.md`). It takes the redacted transcripts produced
by `redact_transcripts.py`, asks Azure `o4-mini` — per transcript, blind, no
gold shown — whether any *complete* direct identifier survived as plain text,
and writes `judge_result_<k>.json` in the exact schema `make_leak_report.py`
aggregates. Run `make_leak_report.py` afterwards for the leak rate + HTML.

The lenient decision rule is the canonical one in `leak_judge_prompt.md`; the
prompt cell below reproduces it for single-transcript scoring.

**Data safety:** this notebook ships with no saved outputs and only synthetic
examples. Real transcripts and the generated HTML stay local and are never
committed.


## 1. Config

In [ ]:
import json
from pathlib import Path

# Which method's redaction to judge. Must match the --method passed to
# redact_transcripts.py. Suffix: '' for finetuned, '_<method>' otherwise.
METHOD = "finetuned"          # finetuned | baseline | rulebased
SUFFIX = "" if METHOD == "finetuned" else f"_{METHOD}"

# Repo-relative paths (notebook lives in evaluation/leak_tests/).
HERE = Path.cwd()
LEAK_DIR = HERE if HERE.name == "leak_tests" else HERE / "evaluation" / "leak_tests"
RESULTS = LEAK_DIR.parent / "results" / "leak_tests"
REDACTED = RESULTS / f"redacted_all{SUFFIX}.jsonl"   # produced by redact_transcripts.py --method
RULES_MD = LEAK_DIR / "leak_judge_prompt.md"          # canonical lenient definition

N_SHARDS = 8          # number of judge_result<suffix>_<k>.json files to emit
print("method              :", METHOD)
print("redacted transcripts:", REDACTED)
print("rules reference     :", RULES_MD)

In [ ]:
from langchain_openai import AzureChatOpenAI

def get_o4_mini():
    """Azure o4-mini judge. Expects the standard AZURE_OPENAI_* environment
    variables (endpoint / API key) to be set, per the company Azure setup."""
    return AzureChatOpenAI(
        azure_deployment="o4-mini",
        api_version="2024-12-01-preview",
        model_name="o4-mini",
        max_completion_tokens=16000,
        timeout=300,          # 5 min
    )

## 2. Load the redacted transcripts

Each line of `redacted_all.jsonl` is `{"line": int, "redacted": str}`.

In [ ]:
transcripts = [json.loads(l) for l in open(REDACTED, encoding="utf-8")]
print(f"{len(transcripts)} redacted transcripts loaded")

## 3. Build the per-transcript judge chain

The prompt reproduces the lenient rule from `leak_judge_prompt.md`, framed to score **one** transcript and return JSON.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.json import JsonOutputParser

JUDGE_PROMPT = """You are a PII residual-leak assessor for a REDACTED call transcript.
Redacted PII already appears as XML-style tags like <FULL_NAME>, <SG_PHONE_NUMBER>,
<SG_ADDRESS>, <SG_NRIC_FIN>, <ACCOUNT_NUMBER>. Decide whether **residual PII** remains
— i.e. a FULL direct identifier still appears as PLAIN, UNTAGGED text.

Count has_residual_pii = true ONLY if you find, as plain untagged text:
- Full NRIC/FIN: a complete 9-character Singapore NRIC (letter + 7 digits + letter,
  e.g. S1234567A). A partial like last-4 "567A" does NOT count.
- Full Mobile Number: a complete contactable number (8 digits for Singapore,
  e.g. 8123-4567 / 91234567, or a full international number). Fragments do NOT count.
- Full Address: block/unit AND street AND postal code together, no partial redaction.
  Isolated components (block only / street only / postal only) do NOT count.

Do NOT count as residual PII (safe or weak):
- Anything already inside a <TAG> (already redacted — exclude entirely).
- Full names, account numbers, company names, job titles.
- Any partial, fragment, or isolated identifier.
- Public business/hotline numbers (e.g. 1-800 toll-free).

Return ONLY this JSON (no prose):
{{"has_residual_pii": <true|false>,
  "details": [{{"type": "Full Mobile"|"Full NRIC"|"Full Address", "value": "<exact plain-text value>"}}]}}

Transcript:
{redacted}
"""

judge_chain = PromptTemplate.from_template(JUDGE_PROMPT) | get_o4_mini() | JsonOutputParser()

## 4. Smoke test on a synthetic transcript

Synthetic PII only — the first mobile is tagged (safe), a second read-back is left in the clear (should be flagged).

In [ ]:
SYNTHETIC = (
    "Agent: Can I take your number?\n"
    "Customer: Sure, it's <SG_PHONE_NUMBER>.\n"
    "Agent: Let me read that back — 9123 4567, correct?\n"      # plain read-back -> should leak
    "Customer: Yes. And my NRIC is <SG_NRIC_FIN>."
)
print(judge_chain.invoke({"redacted": SYNTHETIC}))

## 5. Run the judge over all transcripts → `judge_result<suffix>_<k>.json`

Sharded into `N_SHARDS` result files, resumable, in the exact schema `make_leak_report.py` reads: `{assessed, leaked_lines, details:[{line,type,value}]}`.

In [ ]:
import math, tqdm

RESULTS.mkdir(parents=True, exist_ok=True)
size = math.ceil(len(transcripts) / N_SHARDS)

for k in range(N_SHARDS):
    shard = transcripts[k * size:(k + 1) * size]
    if not shard:
        continue
    out_path = RESULTS / f"judge_result{SUFFIX}_{k}.json"
    if out_path.exists():
        print(f"shard {k}: already done, skipping")
        continue

    leaked_lines, details = [], []
    for t in tqdm.tqdm(shard, desc=f"shard {k}"):
        try:
            v = judge_chain.invoke({"redacted": t["redacted"]})
        except Exception as e:               # keep going; a failed call just scores nothing
            print(f"  line {t['line']} error: {e}")
            continue
        if v.get("has_residual_pii"):
            leaked_lines.append(t["line"])
            for d in v.get("details", []):
                details.append({"line": t["line"], "type": d["type"], "value": d["value"]})

    json.dump({"assessed": len(shard), "leaked_lines": leaked_lines, "details": details},
              open(out_path, "w"))
    print(f"shard {k}: {len(shard)} assessed, {len(leaked_lines)} leaked -> {out_path.name}")

## 6. Aggregate → leak rate + HTML report

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, str(LEAK_DIR / "make_leak_report.py"), "--method", METHOD],
                     capture_output=True, text=True).stdout)
# -> results/reports/leaked_transcripts<suffix>.html (contains real PII; stays local)

---
**Last result (keeper model):** 28/419 files (6.7%), 26/258 customers (10.1%) —
almost all mobile read-backs the model tagged on first mention but missed when
the agent repeated them. See `README.md` in this folder for the full write-up.
